# Performance

Writing code that works is only the first step; writing code that executes quickly and consumes minimal memory is the hallmark of a senior engineer. This phase explores how to analyze, measure, and optimize Python performance.

## 1. Big O Notation

Big O notation is the mathematical framework used to describe the time complexity (how execution time scales with input size $N$) and space complexity (how memory usage scales with input size) of an algorithm.

$O(1)$ - Constant Time: Execution time stays the same regardless of input size (e.g., looking up a dictionary key by hash).

$O(\log N)$ - Logarithmic Time: Halves the search space with each step (e.g., binary search).

$O(N)$ - Linear Time: Execution time scales directly with input size (e.g., looping through an unsorted list).

$O(N \log N)$ - Linearithmic Time: Common in efficient sorting algorithms (e.g., Timsort in Python's sorted()).

$O(N^2)$ - Quadratic Time: Nested loops over the same dataset (e.g., comparing every element in a list against every other element).

In [ ]:
# O(1) - Constant Time Lookup
def get_first_item(items):
    return items[0]

# O(N) - Linear Time Search
def find_item(items, target):
    for item in items:
        if item == target:
            return True
    return False

## 2. The timeit Module
When you want to measure the execution time of small code snippets (like comparing two different ways to write a function), timeit is the built-in gold standard. It runs code repeatedly in a separate loop to minimize the noise caused by background operating system processes.

In [ ]:
import timeit

# Compare list comprehension vs. map()
code_comprehension = """
numbers = range(1000)
result = [x * 2 for x in numbers]
"""

code_map = """
numbers = range(1000)
result = list(map(lambda x: x * 2, numbers))
"""

time_comp = timeit.timeit(stmt=code_comprehension, number=10000)
time_mapped = timeit.timeit(stmt=code_map, number=10000)

print(f"List Comprehension time: {time_comp:.5f} seconds")
print(f"Map time: {time_mapped:.5f} seconds")

## 3. Profiling with cProfile
While timeit handles isolated snippets, cProfile is a built-in deterministic profiler that monitors an entire script, breaking down how much time is spent inside each individual function call.

You can run it directly from your terminal:

In [ ]:
python -m cProfile -s cumulative my_script.py

-s cumulative: Sorts the output by the cumulative time spent in functions, instantly revealing your application's biggest performance bottlenecks.

## 4. Memory Profiling
CPU time isn't the only performance constraint; memory leaks or high RAM usage can crash applications. While Python's standard library doesn't include a built-in memory profiler, the popular third-party package memory_profiler is the standard tool for the job.

Install via pip: pip install memory_profiler

Use the @profile decorator on functions you want to inspect:

In [ ]:
# Run script with: python -m memory_profiler script.py
@profile
def heavy_memory_task():
    large_list = [i for i in range(1000000)]
    return sum(large_list)

if __name__ == "__main__":
    heavy_memory_task()

## 5. Optimization Techniques

Once you identify a bottleneck using profiling tools, you can apply several Python-specific optimization techniques:

Use Built-In Data Structures wisely: Built-in structures like lists, dicts, and sets are written in C under the hood. Always use a set or dict for membership testing ($O(1)$ average time complexity) instead of scanning a list ($O(N)$ time complexity).

Leverage Generators: Instead of loading large datasets entirely into RAM at once using lists, use generator expressions or yield to process data lazily item by item.

Localize Variable Lookups: Python looks up global variables slower than local variables. If you access a global function or variable inside a tight loop, bind it to a local variable first:

In [ ]:
import math

# Slower inside a massive loop
# for x in data: math.sqrt(x)

# Faster optimization
fast_sqrt = math.sqrt
# for x in data: fast_sqrt(x)

**Use C-Extensions / Cython / PyPy:** If pure Python isn't fast enough for heavy number crunching, offload math tasks to optimized libraries like NumPy, compile code using Cython, or run your application using PyPy (an alternative Python interpreter featuring a Just-In-Time compiler).

Best Practices & Common Pitfalls

**Never Prematurely Optimize:** Don't write complex, unreadable code upfront trying to make it "fast." Profile first, find the actual bottleneck, and optimize only where measurements prove it matters.

**Beware of String Concatenation in Loops:** Strings are immutable in Python. Using += inside a loop creates a brand new string object in memory every single iteration, leading to $O(N^2)$ performance. Use "".join(list_of_strings) instead.

**Understand Data Types:** Storing millions of objects in standard Python lists or dictionaries consumes massive amounts of memory due to object overhead. Use NumPy arrays or structured buffers when handling heavy numerical data.